Here is the code where you put in H, M, G, model and measurement

In [66]:
#| default_exp prediction_module

In [16]:
#| export
import pickle
import pandas as pd 
from anthropmass.data_module import *
from anthropmass.anthro_module import *
from anthropmass.bambi_module import *

This functions fetches the pickled model of the desired measurement

In [17]:
#| export
def get_pickled_model(kindofmodel:str, measurement:str):
    filepath = f'../output/anthro_models/{kindofmodel}/pickle_{measurement}_{kindofmodel}'
    with open(filepath,'rb') as file:
        model=pickle.load(file)
    return model

Predict_from_model uses the pickled model and the "minus mean person" to predict the measurement. One of the arguments passed into this functions is "kind of model", and currently there are three different models to chose between. The three models are xgboost, bambi, and bambi with component as input. When using bambi with component as input, the component needs to be passed as an argument into the functions, otherwise Army Reserve is set as default. The three different components to pass into the function are Regular Army, Army National Guard, and Army Reserve.

In [18]:
#| export
def predict_from_model(pickledmodel, kindofmodel:str, measurement:str, input_person:pd.DataFrame, c=False):
    person = minus_mean(input_person,['weightkg','stature'])
    if kindofmodel=='xgboost':
        return pickledmodel.predict(person)
    
    elif kindofmodel=='bambi':
        formula = 'weightkg + stature + C(Gender)'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    elif kindofmodel=='bambi_c':
        if isinstance(c, pd.Series):
            person['Component']=c

        elif c!= False:
            person['Component']=c
        
        else:
            person['Component']='Army Reserve'
        formula='0 + C(Gender) + Component + weightkg + stature'
        model = make_model_formula(measurement, formula)
        return predict_mean_bmb(pickledmodel, model, person, measurement)
    
    else:
        return 'wrong model name'

The function below allows us to obtain the pickled model for the desired measurement and then use this pickled model in the funktion "predict_from_model" where the individuals are predicted with this pickled model. 

In [19]:
def predict_column(kindofmodel:str, measurement:str, group:pd.DataFrame, c=False):
    pickledmodel = get_pickled_model(kindofmodel, measurement)
    preds = predict_from_model(
    pickledmodel,
    kindofmodel,
    measurement,
    group[['weightkg','stature','Gender']], c)

    return preds

This function allows us to loop through the list of desired measurements. For each measurement the function "predict_column" is called upon, which allowes us to predict one measurement for every individual before moving on to the next measurement. 

In [20]:
def loop_measurements(kindofmodel:str, measurements:list, group:pd.DataFrame, c=False):
    preds_all=pd.DataFrame()
    for m in measurements:
        preds_all[m]=predict_column(kindofmodel, m, group, c)
    return preds_all

The code below is an example of a group of individuals (test) where every measurement from ansur can be predicted (all names). 

In [21]:
test= pd.read_csv('../data/processed/ANSURIItest.csv')
all_names= measurement_names()

In [22]:
xgboost=loop_measurements('xgboost', all_names, test)
xgboost.to_csv('../output/anthro_results/predict_all_test_xgboost.csv', index=False)

In [23]:
bambi=loop_measurements('bambi', all_names, test)
bambi.to_csv('../output/anthro_results/predict_all_test_bambi.csv', index=False)

KeyboardInterrupt: 

When using bambi_c you can specify which component the individuals should have. If all individual should have the same component write 'Army Reserve', 'Army National Guard' or 'Regular Army'. If the data set has column with component that you want to use, write yourdataset['Component'] as input. This is shown in the following code cells.

In [ ]:
bambi_c=loop_measurements('bambi_c', all_names, test, test['Component'])
bambi_c.to_csv('../output/anthro_results/predict_all_test_bambi_c.csv', index=False)

In this cell eveybody is predicted as Army Reserve

In [ ]:
bambi_c_AR=loop_measurements('bambi_c', all_names, test, 'Army Reserve')
bambi_c_AR.to_csv('../output/anthro_results/predict_all_test_bambi_c_AR.csv', index=False)